In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)
import json
import numpy as np

from leadgate.data import load_preprocessed
from leadgate.pipeline import make_champion_pipeline
from leadgate.threshold import calculate_profit

In [ ]:
X_train, X_test, y_train, y_test = load_preprocessed("../data/processed")

In [ ]:
log_reg_pipe = make_champion_pipeline()

with open("../models/threshold.json", "r") as f:
    data = json.load(f)

threshold = data["threshold"]
print(threshold)

In [ ]:
log_reg_pipe.fit(X_train, y_train)
proba = log_reg_pipe.predict_proba(X_test)[:, 1]
print(proba)

In [ ]:
prec, rec, thr = precision_recall_curve(y_test, proba)
ap = average_precision_score(y_test, proba)
plt.plot(rec, prec, label=f"Average precision score: {ap:.3f}")
plt.axhline(0.117, ls="--", color="gray", label="base rate = 0.117")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.title("PR-AUC Curve")
plt.savefig("../reports/figures/test-PR-AUC.png")
plt.show()

In [ ]:
fpr, tpr, thr = roc_curve(y_test, proba)
roc_auc = roc_auc_score(y_test, proba)
plt.plot(fpr, tpr, label=f"Model ROC-AUC score: {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], label="no-skill: 0.5", ls="--")
plt.ylabel("True Positive Rate")
plt.xlabel("False Positive Rate")
plt.legend()
plt.title("ROC-AUC Curve")
plt.savefig("../reports/figures/test-ROC-AUC.png")
plt.show()

In [ ]:
y_pred = (proba >= threshold).astype(int)
print(y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
confusion_matrix(y_test, y_pred)

In [ ]:
money = calculate_profit(y_test, y_pred)
print(money)

A threshold tuned on out-of-fold data transferred to the untouched test at 0.69 recall / 0.24 precision against 0.67 / 0.24 predicted, earning €4.75/lead against a €4.61 forecast.

In [ ]:
model = log_reg_pipe["model"]
cols = log_reg_pipe["preprocessor"].get_feature_names_out()
coefs = pd.Series(np.exp(model.coef_[0]), cols).sort_values()
print(f"model intercept: {float(model.intercept_[0])}")
print(coefs)

We can see that `poutcome` = success increases the odds of the "yes". Same with a few months, `job` = retired, `contact` = cellular. Columns like `contact` = unknown, `poutcome` = failure/unknown, `housing` = yes, `loan` = yes, `default` = yes will decrease the odds of the "yes".

In [ ]:
joblib.dump(log_reg_pipe, "../models/model.joblib")

Now I am ready to start the AWS part. In conclusion, I chose and trained Logistic Regression over Gradient Boosting even though boosting had slightly better performance. On the other hand Logistic is a lightweight model (a few KB vs a few MB with GB), faster than GB and interpretable, which we can use later to explain why this customer will subscribe.